# Fidelidade do Grad-CAM (Insertion/Deletion AUC) — reavaliação

Recalcula a fidelidade do Grad-CAM **sem retreinar**: só precisa do `.keras` de cada modelo (anexado como Model,
Dataset ou Output de uma versão salva) e do dataset de teste. Gera `outputs/{Modelo}_{dataset}/gradcam_fidelidade_{Modelo}_{dataset}.csv`
no mesmo formato do `mobilenet.ipynb`, com colunas extras.

**Por que reavaliar** — a versão anterior da métrica tinha dois problemas:

1. **Substrato da remoção.** O `mobilenet.ipynb` usava a foto borrada tanto na inserção quanto na remoção. O borrão
   preserva cor e manchas grandes, então para ferrugem/cercosporiose a confiança nem caía (no EfficientNetV2B0 × BRACOL:
   remoção de 1,000 na ferrugem e 0,994 na classe saudável) e a métrica não discriminava nada. O RISE original
   (Petsiuk, Das & Saenko, 2018) usa **imagem borrada como ponto de partida da inserção** e **pixels pretos (zeros) na
   remoção** — é o que este notebook usa.
2. **Grad-CAM do MobileNetV2/NASNetMobile.** No Keras 3, o `preprocess_input` (x/127,5 − 1) não é uma camada e o laço
   do `make_gradcam_heatmap` antigo o pulava: o backbone recebia pixels 0–255 e o mapa saía errado (erro máximo de
   1,0 contra o gradiente exato). EfficientNetV2/MobileNetV3 normalizam dentro do backbone e não eram afetados.
   Este notebook usa o Grad-CAM corrigido (o mesmo que agora está no `mobilenet.ipynb`).

**Linha de base aleatória.** Cada imagem também é avaliada com um mapa **aleatório** (mesma grade). Uma explicação só
é útil se o score (Inserção − Remoção) dela for **maior que o do aleatório** — o teste de Wilcoxon pareado no final
responde exatamente isso.

In [ ]:
import os
import glob
import json
import zipfile

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy import stats

print(f"TensorFlow: {tf.__version__}")

# ── Configuração ────────────────────────────────────────────────────────────────────
N_POR_CLASSE = 20        # imagens de teste CORRETAMENTE classificadas por classe (o mobilenet.ipynb usava 3)
BLUR_KSIZE = 31          # mesmo borrão do mobilenet.ipynb (ponto de partida da inserção)
N_ALEATORIOS = 3         # mapas aleatórios por imagem (a média deles é a linha de base)
SEED = 42
IMG_SIZE = 224

# Mesmos caminhos do mobilenet.ipynb; se não existirem, o notebook procura em /kaggle/input uma pasta com
# train/val/test com o número de classes do modelo.
DATASETS = {
    'bracol':               '/kaggle/input/newdataset/new_dataset',
    'rocole':               '/kaggle/working/datasets/rocole',
    'coffee_leaf_diseases': '/kaggle/working/datasets/coffee_leaf_diseases',
    'jmuben':               '/kaggle/working/datasets/jmuben',
}

# Todos os .keras anexados (nome no formato {Modelo}_{dataset}.keras, como o mobilenet.ipynb salva)
MODELOS = sorted(set(
    glob.glob('/kaggle/input/**/*.keras', recursive=True) + glob.glob('outputs/*/*.keras')
))
print(f"{len(MODELOS)} modelo(s) encontrado(s):")
for m in MODELOS:
    print(f"  {m}")
if not MODELOS:
    raise FileNotFoundError("Nenhum .keras encontrado. Anexe o(s) modelo(s) treinado(s) (Add Input → Models/Datasets/Notebooks).")

In [ ]:
def _n_classes(pasta):
    return sum(os.path.isdir(os.path.join(pasta, d)) for d in os.listdir(pasta))


def localizar_teste(dataset, n_classes):
    """Pasta test/ do dataset: usa DATASETS se existir; senão procura em /kaggle/input."""
    base = DATASETS.get(dataset)
    if base and os.path.isdir(os.path.join(base, 'test')):
        return os.path.join(base, 'test')
    for train in sorted(glob.glob('/kaggle/input/**/train', recursive=True)):
        raiz = os.path.dirname(train)
        if os.path.isdir(os.path.join(raiz, 'test')) and _n_classes(train) == n_classes:
            return os.path.join(raiz, 'test')
    raise FileNotFoundError(f"Pasta test/ de '{dataset}' com {n_classes} classes não encontrada em /kaggle/input.")


_extratores_gradcam = {}


def make_gradcam_heatmap(img_array, model, pred_index=None):
    """Grad-CAM corrigido -- idêntico ao do mobilenet.ipynb (ver a célula markdown do topo):
    extrator com o pré-processamento incluído, training=False explícito e derivada estável da softmax."""
    if id(model) not in _extratores_gradcam:
        _extratores_gradcam[id(model)] = tf.keras.Model(model.inputs, model.get_layer('global_avg_pool').input)
    extrator = _extratores_gradcam[id(model)]
    idx_gap = model.layers.index(model.get_layer('global_avg_pool'))
    W_final, b_final = [tf.constant(v, tf.float32) for v in model.get_layer('predicao_final').get_weights()]

    with tf.GradientTape() as tape:
        feature_maps = extrator(img_array, training=False)
        tape.watch(feature_maps)
        out = feature_maps
        for layer in model.layers[idx_gap:-1]:
            out = layer(out, training=False)
        penultima = tf.cast(out, tf.float32)

    p = tf.nn.softmax(penultima @ W_final + b_final)[0]
    if pred_index is None:
        pred_index = int(tf.argmax(p))
    W_t = tf.transpose(W_final)
    dp_dpenultima = p[pred_index] * tf.reduce_sum(p[:, None] * (W_t[pred_index][None, :] - W_t), axis=0)
    grads = tape.gradient(penultima, feature_maps, output_gradients=dp_dpenultima[None, :])

    pooled_grads = tf.reduce_mean(tf.cast(grads, tf.float32), axis=(0, 1, 2))
    heatmap = tf.squeeze(tf.cast(feature_maps[0], tf.float32) @ pooled_grads[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-12)
    return heatmap.numpy()


def carregar(caminho):
    img = tf.io.decode_image(tf.io.read_file(caminho), channels=3, expand_animations=False)
    return tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method='bilinear').numpy().astype(np.float32)


_trapz = getattr(np, 'trapezoid', np.trapz)


def curva(model, img_rgb, mapa, classe, modo):
    """Inserção: parte da imagem borrada e revela as células do mapa da mais para a menos importante.
    Remoção: parte da imagem original e pinta as células de PRETO na mesma ordem. Devolve (probs, AUC)."""
    H, W = img_rgb.shape[:2]
    hh, hw = mapa.shape
    ordem = np.argsort(mapa.ravel(), kind='stable')[::-1]
    bh, bw = H // hh, W // hw
    blur = cv2.GaussianBlur(img_rgb, (BLUR_KSIZE, BLUR_KSIZE), 0)
    base = blur if modo == 'insercao' else img_rgb
    fonte = img_rgb if modo == 'insercao' else np.zeros_like(img_rgb)

    atual = base.copy()
    frames = [atual.copy()]
    for idx in ordem:
        r, c = divmod(int(idx), hw)
        atual[r * bh:min((r + 1) * bh, H), c * bw:min((c + 1) * bw, W)] = \
            fonte[r * bh:min((r + 1) * bh, H), c * bw:min((c + 1) * bw, W)]
        frames.append(atual.copy())
    probs = model.predict(np.stack(frames), verbose=0, batch_size=64)[:, classe]
    return probs, float(_trapz(probs, np.linspace(0, 1, len(probs))))

# Avaliação

In [ ]:
rng = np.random.default_rng(SEED)
resultados = []
curvas_medias = {}

for caminho_modelo in MODELOS:
    nome = os.path.splitext(os.path.basename(caminho_modelo))[0]
    modelo_nome, _, dataset = nome.partition('_')
    print(f"\n{'=' * 80}\n  {nome}\n{'=' * 80}")
    tf.keras.backend.clear_session()
    _extratores_gradcam.clear()
    model = tf.keras.models.load_model(caminho_modelo, compile=False)
    n_classes = model.output_shape[-1]

    pasta_teste = localizar_teste(dataset, n_classes)
    ds = tf.keras.utils.image_dataset_from_directory(
        pasta_teste, image_size=(IMG_SIZE, IMG_SIZE), batch_size=32, label_mode='int', shuffle=False
    )
    classes = ds.class_names
    # Só as predições ficam na memória (o teste do JMuBEN inteiro em float32 passaria de 5 GB); as imagens
    # selecionadas são relidas do disco com o mesmo decode + resize bilinear do image_dataset_from_directory.
    y_true = np.concatenate([y.numpy() for _, y in ds])
    probs_teste = model.predict(ds, verbose=0)
    y_pred = probs_teste.argmax(1)
    print(f"Teste: {len(y_true)} imagens | acurácia {np.mean(y_true == y_pred):.4f}")

    acum = {k: [] for k in ('gc_ins', 'gc_del', 'al_ins', 'al_del')}
    for classe, nome_classe in enumerate(classes):
        indices = np.where((y_true == classe) & (y_pred == classe))[0][:N_POR_CLASSE]
        for i in indices:
            img = carregar(ds.file_paths[i])
            mapa = make_gradcam_heatmap(img[None], model, classe)
            p_ins, auc_ins = curva(model, img, mapa, classe, 'insercao')
            p_del, auc_del = curva(model, img, mapa, classe, 'remocao')
            acum['gc_ins'].append(p_ins); acum['gc_del'].append(p_del)

            aleat_ins, aleat_del = [], []
            for _ in range(N_ALEATORIOS):
                mapa_aleatorio = rng.random(mapa.shape)
                pi, ai = curva(model, img, mapa_aleatorio, classe, 'insercao')
                pd_, ad = curva(model, img, mapa_aleatorio, classe, 'remocao')
                aleat_ins.append(ai); aleat_del.append(ad)
                acum['al_ins'].append(pi); acum['al_del'].append(pd_)

            resultados.append({
                'modelo': modelo_nome, 'dataset': dataset, 'classe': nome_classe,
                'arquivo': os.path.basename(ds.file_paths[i]),
                'confianca': float(probs_teste[i, classe]),
                'insertion_auc': auc_ins, 'deletion_auc': auc_del, 'score_fidelidade': auc_ins - auc_del,
                'insertion_auc_aleatorio': float(np.mean(aleat_ins)),
                'deletion_auc_aleatorio': float(np.mean(aleat_del)),
                'score_aleatorio': float(np.mean(aleat_ins) - np.mean(aleat_del)),
            })
        print(f"  [{nome_classe}] {len(indices)} imagem(ns)")
    curvas_medias[nome] = {k: np.mean(v, axis=0) for k, v in acum.items()}

    df_modelo = pd.DataFrame([r for r in resultados if f"{r['modelo']}_{r['dataset']}" == nome])
    os.makedirs(os.path.join('outputs', nome), exist_ok=True)
    df_modelo.to_csv(os.path.join('outputs', nome, f'gradcam_fidelidade_{nome}.csv'), index=False)
    del model

# Resultados

In [ ]:
df = pd.DataFrame(resultados)
df['experimento'] = df['modelo'] + ' / ' + df['dataset']

colunas = ['insertion_auc', 'deletion_auc', 'score_fidelidade', 'insertion_auc_aleatorio',
           'deletion_auc_aleatorio', 'score_aleatorio']
print("── Média por classe ──")
print(df.groupby(['experimento', 'classe'])[colunas].mean().round(3).to_string())

linhas = []
for exp, g in df.groupby('experimento'):
    # Wilcoxon pareado (mesma imagem): o score do Grad-CAM é maior que o do mapa aleatório?
    dif = g['score_fidelidade'] - g['score_aleatorio']
    p = stats.wilcoxon(g['score_fidelidade'], g['score_aleatorio'], alternative='greater').pvalue if (dif != 0).any() else np.nan
    linhas.append({
        'experimento': exp, 'n_imagens': len(g),
        'insertion_auc': g['insertion_auc'].mean(), 'deletion_auc': g['deletion_auc'].mean(),
        'score': g['score_fidelidade'].mean(), 'score_aleatorio': g['score_aleatorio'].mean(),
        'ganho_sobre_aleatorio': dif.mean(), 'p_wilcoxon': p,
        'gradcam_melhor_que_aleatorio_5%': 'Sim' if p < 0.05 else 'Não',
    })
df_resumo = pd.DataFrame(linhas)
print("\n── Resumo por modelo ──")
print(df_resumo.round(4).to_string(index=False))
df_resumo.to_csv(os.path.join('outputs', 'gradcam_fidelidade_resumo.csv'), index=False)

In [ ]:
fig, axes = plt.subplots(len(curvas_medias), 2, figsize=(13, 4.2 * len(curvas_medias)), squeeze=False)
for linha, (nome, c) in zip(axes, curvas_medias.items()):
    x = np.linspace(0, 100, len(c['gc_ins']))
    for ax, chave, titulo in [(linha[0], 'ins', 'Inserção (maior AUC é melhor)'), (linha[1], 'del', 'Remoção (menor AUC é melhor)')]:
        ax.plot(x, c[f'gc_{chave}'], linewidth=2.2, label=f"Grad-CAM (AUC={_trapz(c[f'gc_{chave}'], x / 100):.3f})")
        ax.plot(x, c[f'al_{chave}'], '--', linewidth=1.8, color='gray',
                label=f"Aleatório (AUC={_trapz(c[f'al_{chave}'], x / 100):.3f})")
        ax.set_title(f'{nome} — {titulo}', fontsize=11)
        ax.set_xlabel('% da imagem revelada' if chave == 'ins' else '% da imagem apagada')
        ax.set_ylabel('Probabilidade da classe')
        ax.set_ylim(0, 1.02)
        ax.grid(True, alpha=0.3)
        ax.legend()
plt.tight_layout()
plt.savefig(os.path.join('outputs', 'gradcam_fidelidade_curvas.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

with zipfile.ZipFile('fidelidade_xai.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob('outputs/**/gradcam_fidelidade_*', recursive=True):
        z.write(f)
print("Baixe fidelidade_xai.zip (painel Output → ⋮ → Download) -- tem os CSVs e o gráfico.")